# Movie Success Prediction
**Gexton Education — Summer Internship Program — Data Science Task #12**

This notebook walks through the full pipeline: data loading, cleaning, exploratory data analysis, model training, evaluation, and prediction on new movie records.

All paths in this notebook are anchored to the project root so the notebook runs correctly regardless of the current working directory.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_style('whitegrid')
%matplotlib inline

## 1. Project Paths

File-anchored paths ensure this notebook works when opened from any location.

In [ ]:
# Project root is one level up from the notebooks/ folder
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
CHARTS_DIR = os.path.join(BASE_DIR, 'charts')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

print('Base directory:', BASE_DIR)
print('Data directory:', DATA_DIR)

## 2. Data Loading & Cleaning

In [ ]:
raw_path = os.path.join(DATA_DIR, 'movies_dataset.csv')
df = pd.read_csv(raw_path)
print('Shape:', df.shape)
df.head()

In [ ]:
# Check for duplicates and missing values
print('Duplicate movie_id rows:', df.duplicated(subset=['movie_id']).sum())
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
# Remove duplicates and impute missing values with the median
df = df.drop_duplicates(subset=['movie_id']).reset_index(drop=True)

numeric_cols_to_impute = ['critic_score', 'social_media_buzz', 'director_popularity']
for col in numeric_cols_to_impute:
    df[col] = df[col].fillna(df[col].median())

print('Shape after cleaning:', df.shape)
print('Remaining missing values:', df.isnull().sum().sum())

## 3. Exploratory Data Analysis

In [ ]:
df['is_success'].value_counts(normalize=True).rename('proportion')

In [ ]:
plt.figure(figsize=(9,5))
order = df['genre'].value_counts().index
sns.countplot(data=df, y='genre', order=order, hue='genre', palette='viridis', legend=False)
plt.title('Movie Count by Genre')
plt.xlabel('Number of Movies')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='budget_million', y='revenue_million', hue='is_success', palette={0:'#e74c3c', 1:'#2ecc71'}, alpha=0.6)
plt.title('Budget vs. Revenue')
plt.xlabel('Budget (Million USD)')
plt.ylabel('Revenue (Million USD)')
plt.show()

In [ ]:
genre_success = df.groupby('genre')['is_success'].mean().sort_values(ascending=False)
plt.figure(figsize=(9,5))
sns.barplot(x=genre_success.values, y=genre_success.index, hue=genre_success.index, palette='viridis', legend=False)
plt.title('Success Rate by Genre')
plt.xlabel('Success Rate')
plt.show()
genre_success

In [ ]:
numeric_df = df.select_dtypes(include=['number']).drop(columns=['movie_id'])
plt.figure(figsize=(11,8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## 4. Feature Preparation

**Note on target leakage:** `audience_rating` is deliberately excluded from the model's feature set. The `is_success` label was partly defined using audience rating, so including it as a predictor would leak target information and produce unrealistically perfect scores. It remains in the dataset for reporting and EDA purposes only.

In [ ]:
TARGET_COL = 'is_success'
NUMERIC_FEATURES = ['release_year', 'budget_million', 'marketing_spend_million', 'runtime_minutes',
                     'cast_popularity', 'director_popularity', 'num_screens', 'critic_score', 'social_media_buzz']
CATEGORICAL_FEATURES = ['genre', 'director_tier', 'studio_type']

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)
])

## 5. Model Training & Evaluation

We train three models — Logistic Regression, Random Forest, and Decision Tree — and tune each with `GridSearchCV`.

In [ ]:
candidates = {
    'Logistic Regression': (LogisticRegression(max_iter=2000, random_state=42),
                             {'classifier__C': [0.01, 0.1, 1, 10], 'classifier__solver': ['lbfgs', 'liblinear']}),
    'Random Forest': (RandomForestClassifier(random_state=42),
                       {'classifier__n_estimators': [200, 400], 'classifier__max_depth': [None, 10, 20], 'classifier__min_samples_split': [2, 5]}),
    'Decision Tree': (DecisionTreeClassifier(random_state=42),
                       {'classifier__max_depth': [5, 10, 15, None], 'classifier__min_samples_split': [2, 5, 10]})
}

results = []
fitted_pipelines = {}

for name, (estimator, param_grid) in candidates.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', estimator)])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_pipe = grid.best_estimator_
    y_pred = best_pipe.predict(X_test)
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred)
    }
    results.append(metrics)
    fitted_pipelines[name] = best_pipe
    print(f'--- {name} (best params: {grid.best_params_}) ---')
    print(classification_report(y_test, y_pred, target_names=['Not Successful', 'Successful']))

In [ ]:
results_df = pd.DataFrame(results).sort_values('f1_score', ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_pipeline = fitted_pipelines[best_model_name]
print('Best model:', best_model_name)

In [ ]:
y_pred_best = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Successful','Successful'], yticklabels=['Not Successful','Successful'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 6. Feature Importance

In [ ]:
classifier = best_pipeline.named_steps['classifier']
cat_names = best_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = np.array(NUMERIC_FEATURES + list(cat_names))

if hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
else:
    importances = np.abs(classifier.coef_[0])

importance_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(9,7))
sns.barplot(data=importance_df, x='importance', y='feature', hue='feature', palette='viridis', legend=False)
plt.title(f'Top Feature Importance — {best_model_name}')
plt.tight_layout()
plt.show()

## 7. Save the Best Model

In [ ]:
model_path = os.path.join(MODELS_DIR, 'best_model.pkl')
joblib.dump(best_pipeline, model_path)
print('Model saved to:', model_path)

## 8. Predicting on New Movie Records

In [ ]:
new_movie = pd.DataFrame([{
    'genre': 'Sci-Fi', 'release_year': 2026, 'budget_million': 150, 'marketing_spend_million': 55,
    'runtime_minutes': 132, 'cast_popularity': 85, 'director_popularity': 78, 'director_tier': 'A-List',
    'studio_type': 'Major Studio', 'num_screens': 4200, 'critic_score': 74, 'social_media_buzz': 88
}])

prediction = best_pipeline.predict(new_movie)[0]
probability = best_pipeline.predict_proba(new_movie)[0, 1]
print('Predicted success:', 'Successful' if prediction == 1 else 'Not Successful')
print(f'Success probability: {probability:.1%}')

## 9. Summary

- Three classification models were trained and compared: Logistic Regression, Random Forest, and Decision Tree.
- The best-performing model was selected based on F1-score and saved with `joblib` for reuse in the Streamlit app.
- Genre, director tier, studio type, marketing spend, and screen count were among the most influential factors in predicting commercial movie success.
- See `report/business_insights.txt` for detailed business recommendations.